In [ ]:
## import packages
from corgisim import scene
from corgisim import instrument
import matplotlib.pyplot as plt
import numpy as np
import proper
from corgisim import outputs

from corgihowfsc.utils import onboard_processing

In [ ]:
#First, import the module:
import roman_preflight_proper
### Then, run the following command to copy the default prescription file 
roman_preflight_proper.copy_here()

In [ ]:
# --- Host Star Properties ---
Vmag = 2.25                        # V-band magnitude of the host star
sptype = 'O5'                      # Spectral type of the host star
ref_flag = False                    # if the target is a reference star or not, default is False
host_star_properties = {'Vmag': Vmag,
                        'spectral_type': sptype,
                        'magtype': 'vegamag',
                        'ref_flag': False}

# Construct a list of dictionaries for all companion point sources
point_source_info = [
]

# --- Create the Astrophysical Scene ---
# This Scene object combines the host star and companion(s)
base_scene = scene.Scene(host_star_properties, point_source_info)

# --- Access the generated stellar spectrum ---
sp_star = base_scene.stellar_spectrum
# --- Access the generated companion spectrum ---
sp_comp = base_scene.off_axis_source_spectrum 


In [ ]:
cgi_mode = 'excam'
cor_type = 'hlc'
bandpass = '1A'

cases = ['3e-8']       
rootname = 'hlc_ni_' + cases[0]

dm1 = proper.prop_fits_read( roman_preflight_proper.lib_dir + '/examples/'+ rootname + '_dm1_v.fits' )
dm2 = proper.prop_fits_read( roman_preflight_proper.lib_dir + '/examples/'+ rootname + '_dm2_v.fits' )


In [ ]:
##  Define the polaxis parameter. Use 10 for non-polaxis cases only, as other options are not yet implemented.
polaxis = 10
# output_dim define the size of the output image
output_dim = 153
# roll_angle in degree, define the roll angle of the telescope.
# roll_angle is defined as the rotation angle of the excam coordinates (X, Y) relative to the sky coordinates(RA,DEC), positive is counter-clockwise
# Default is 0 degrees, corresponding to North up, East left in the sky coordinates.
roll_angle = 0

### define a dictinatary to pass keywarod to proper
### optics_keywords are the keyword arguments to the internal functions for package proper, which define the optics of coronagraph
# use_dm1/use_dm2: if use dm
# use_fpm: if use focal plane mask
# use_lyot_stop: if use lyot stop 
# use_field_stop: if use field stop
# other paramters that could pass to Proper defined by CgiSim
nd_filter = 0
##options for nd_filter:
##nd_filter = 1: ND 2.25 @ FPAM
##nd_filter = 2: ND 4.75 @ FPAM
##nd_filter = 3: ND 4.75 @ FSAM

optics_keywords ={'cor_type':cor_type, 'use_errors':2, 'polaxis':polaxis, 'output_dim':output_dim,\
                    'use_dm1':1, 'dm1_v':dm1, 'use_dm2':1, 'dm2_v':dm2,'use_fpm':1, 'use_lyot_stop':1,  'use_field_stop':1 ,"nd":nd_filter}

##visit_type and visit_id are populated into the headers
visit_type = 'CGIVST_TDD_OBS'
visit_id  = '0020001001001901001'

##define the corgi.optics class that hold all information about the instrument paramters                    
optics = instrument.CorgiOptics(cgi_mode, bandpass, optics_keywords=optics_keywords, if_quiet=True,roll_angle=roll_angle,
                                visit_type=visit_type,visit_id=visit_id )

In [ ]:
## Pass the base_scene object to corgi.optics and use get_psf to simulate the host star PSF.
## The result is stored in a SimulatedImage object as an Astropy HDU containing both data and header information.

sim_scene = optics.get_host_star_psf(base_scene)
image_star_corgi = sim_scene.host_star_image.data

In [ ]:
import eetc
from eetc.cgi_eetc import CGIEETC
import os

eetc_path = os.path.dirname(os.path.abspath(eetc.__file__))
seq = 'OPEN_NFOV_1A_SEED'


If using EETC

In [ ]:
get_cgi_eetc = CGIEETC(mag=Vmag,
                       phot='v', # only using V-band magnitudes as a standard
                       spt=sptype, # only using G0V as a standard
                       pointer_path=os.path.join(eetc_path, 'pointer_howfsc.yaml'))

In [ ]:
unprobed_snr = 10

scale = 1e-5
scale_bright = 1e-4

nframes, exptime, gain, snr_out, optflag = \
    get_cgi_eetc.calc_exp_time(
        sequence_name=seq,
        snr=unprobed_snr,
        scale=scale,
        scale_bright=scale_bright,
    )

print(f"nframes: {nframes}, exptime: {exptime}, gain: {gain}, snr_out: {snr_out}, optflag: {optflag}")

In [ ]:

### emccd_keywords are the keyword arguments to the internal functions for  emccd_detect,
###  and that everything stays the default settings unless otherwise changed.
### In this example, we'll use the default parameters for the EMCCD detector, except for the EM gain.

emccd_keywords ={'em_gain': 3000, 'cr_rate': 10}
exptime = 5
detector = instrument.CorgiDetector(emccd_keywords, photon_counting = True)

## the default is photon_counting = False, which will set header ISPC=0, which means the output is in analog mode. 
# If detector = instrument.CorgiDetector(emccd_keywords, photon_counting = True), then ISPC=1, which means the output is in photon counting mode.
# it will not change the simulation, but only change the header keyword ISPC in the output fits file


In [ ]:

#In real observations, exposures are typically broken into a sequence of short frames (e.g., 100s per frame) to reduce the impact of cosmic ray hits.
#However, for simplicity in this example, we'll simulate a single long exposure (10000s) here.

sim_scene = detector.generate_detector_image(sim_scene, exptime)
image_tot_corgi_sub= sim_scene.image_on_detector.data

In [ ]:
plt.imshow(image_tot_corgi_sub,origin='lower')
plt.title('Combined Image with detector noise, CorgiSim')

co = plt.colorbar(shrink=0.7)

In [ ]:
def generate_master_dark(detector, exptime):
    """
    dark:  master dark
    FPM: fixed pattern noise map
    gain: EM gain
    exptime: exposure time
    D: dark current rate map
    C: CIC map
    """
    D = detector.emccd.dark_current * np.ones((output_dim, output_dim))
    C = detector.emccd.cic * np.ones((output_dim, output_dim))
    FPN = np.zeros((output_dim, output_dim)) # Not included in emccd_detect
    dark = FPN / detector.emccd.em_gain + exptime * D + C

    return dark

In [ ]:
master_dark = generate_master_dark(detector, exptime)
B = detector.emccd.bias * np.ones((output_dim, output_dim))

In [ ]:
# Keep the raw detector output in DN for onboard processing.
raw_frames_dn = []
nframes = 20
for _ in range(nframes):
    sim_scene = detector.generate_detector_image(sim_scene, exptime)
    raw_frames_dn.append(sim_scene.image_on_detector.data.copy())
raw_frames_dn = np.stack(raw_frames_dn)

Cosmic-ray detection operates on each raw DN frame after one bias subtraction. The row filter detects **saturated multi-pixel plateaus** and masks from the plateau's leading edge to the end of that row. Bright but unsaturated hits may remain. No flat field is applied here.

In [ ]:
bias_e = detector.emccd.bias
e_per_dn = detector.emccd.eperdn
em_gain = detector.emccd.em_gain
full_well_image_e = detector.emccd.full_well_image
full_well_serial_e = detector.emccd.full_well_serial
master_dark_e = generate_master_dark(detector, exptime)

result = onboard_processing.process_onboard_frames(
    raw_frames_dn,
    bias_e=bias_e,
    e_per_dn=e_per_dn,
    em_gain=em_gain,
    full_well_image_e=full_well_image_e,
    full_well_serial_e=full_well_serial_e,
    master_dark_e=master_dark_e,
    cosmic_filter_width=2,
    saturation_threshold=0.99,
    plateau_threshold=0.85,
)

calibrated = result.image
bias_subtracted_dn = raw_frames_dn.astype(float) - bias_e / e_per_dn
masked_frames_dn = np.where(result.bad_pixel_mask, np.nan, bias_subtracted_dn)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))

im0 = ax[0].imshow(bias_subtracted_dn[0], origin='lower')
ax[0].set_title('Bias Subtracted Image')
fig.colorbar(im0, ax=ax[0], shrink=0.8, label='DN')

im1 = ax[1].imshow(result.cosmic_ray_mask[0], origin='lower', vmin=0, vmax=1, cmap='gray')
ax[1].set_title('Cosmic ray mask')
fig.colorbar(im1, ax=ax[1], shrink=0.8, label='Mask')

plt.tight_layout()
plt.show()

In [ ]:
full_well_dn = min(full_well_image_e * em_gain, full_well_serial_e) / e_per_dn
print(f"Saturation detection threshold: {0.99 * full_well_dn:.1f} bias-subtracted DN")
for i, frame in enumerate(bias_subtracted_dn):
    print(f"Frame {i + 1}: maximum {frame.max():.1f} DN, "
          f"masked {result.cosmic_ray_mask[i].sum()} pixels in "
          f"{np.any(result.cosmic_ray_mask[i], axis=1).sum()} rows")

In [ ]:
# The mask panel shows rejected pixels explicitly; subtracting NaN-masked
# frames from the originals would only produce zeros and NaNs.
nshow = min(3, len(raw_frames_dn))
fig, ax = plt.subplots(3, nshow, figsize=(4 * nshow, 10), squeeze=False,
                       constrained_layout=True)
for i in range(nshow):
    ax[0, i].imshow(raw_frames_dn[i], origin='lower')
    ax[0, i].set_title(f'Raw Frame {i + 1}')
    ax[1, i].imshow(masked_frames_dn[i], origin='lower')
    ax[1, i].set_title(f'Frame masked with bad pixels map {i + 1}')
    ax[2, i].imshow(result.cosmic_ray_mask[i], origin='lower',
                    vmin=0, vmax=1, cmap='gray')
    ax[2, i].set_title(f'Cosmic-ray mask {i + 1}')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
ax[0].imshow(calibrated, origin='lower')
ax[0].set_title('Combined, dark-subtracted image (electrons)')
ax[1].imshow(result.good_frame_count, origin='lower', vmin=0,
             vmax=len(raw_frames_dn), cmap='viridis')
ax[1].set_title('Good frames per pixel')
plt.show()

In [ ]:
result_3 = onboard_processing.process_onboard_frames(
    raw_frames_dn[:3],
    bias_e=bias_e,
    e_per_dn=e_per_dn,
    em_gain=em_gain,
    full_well_image_e=full_well_image_e,
    full_well_serial_e=full_well_serial_e,
    master_dark_e=master_dark_e,
    cosmic_filter_width=2,
    saturation_threshold=0.99,
    plateau_threshold=0.85,
)

calibrated = result_3.image

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
ax[0].imshow(calibrated, origin='lower')
ax[0].set_title('Combined, dark-subtracted image (electrons)')
ax[1].imshow(result.good_frame_count, origin='lower', vmin=0,
             vmax=len(raw_frames_dn), cmap='viridis')
ax[1].set_title('Good frames per pixel')
plt.show()

In [ ]:
result_5 = onboard_processing.process_onboard_frames(
    raw_frames_dn[:3],
    bias_e=bias_e,
    e_per_dn=e_per_dn,
    em_gain=em_gain,
    full_well_image_e=full_well_image_e,
    full_well_serial_e=full_well_serial_e,
    master_dark_e=master_dark_e,
    cosmic_filter_width=2,
    saturation_threshold=0.99,
    plateau_threshold=0.85,
)

calibrated = result_5.image

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
ax[0].imshow(calibrated, origin='lower')
ax[0].set_title('Combined, dark-subtracted image (electrons)')
ax[1].imshow(result.good_frame_count, origin='lower', vmin=0,
             vmax=len(raw_frames_dn), cmap='viridis')
ax[1].set_title('Good frames per pixel')
plt.show()